# kompyle — Knowledge Compilation into *klay* Circuits

> **Repository:** https://github.com/ML-KULeuven/kompyle  
> **PyPI:** `pip install kompyle`

This notebook walks through the full public API of **kompyle**

---
## 0. Installation

**From wheel (recommended):**
```bash
pip install kompyle
```

> **Note:** the wheel bundles Ganak, Arjun, d4v2 and all native dependencies.
> SDD support requires the optional extra `pip install kompyle[sdd]`

**From source (full dev build):**
```bash
git clone https://github.com/ML-KULeuven/kompyle.git
cd kompyle
podman-compose build
podman-compose up -d dev
podman-compose exec dev bash
python -m venv .venv && source .venv/bin/activate
pip install -e '.[dev]'
```

In [32]:
!pip install kompyle


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [33]:
import kompyle as p

print('kompyle version :', p.__version__)
print('public API:', [s for s in dir(p) if not s.startswith('_')])

kompyle version : 0.0.1.dev98
public API      : ['ArjunOptions', 'Circuit', 'D4Options', 'D4PreprocMethod', 'D4Solver', 'GanakOptions', 'GatedFormula', 'NodePtr', 'PackageNotFoundError', 'SDNNFResult', 'SDNNFViolation', 'annotations', 'check_decomposability', 'check_sdnnf', 'check_smooth', 'compile_from_cnf_using_d4v2', 'compile_from_cnf_using_ganak', 'compile_from_cnf_using_sdd', 'compile_from_gates_file_using_d4v2', 'compile_from_gates_formula_using_d4v2', 'compile_from_sdd', 'count_from_cnf_using_d4v2', 'count_from_cnf_using_ganak', 'count_from_cnf_using_sdd', 'ctypes', 'get_d4_stats', 'get_ganak_stats', 'glob', 'os', 'pkompyle', 'reset_all_stats', 'reset_d4_stats', 'reset_ganak_stats', 'sys', 'version']


### Helper: writing DIMACS CNF files

We reuse this throughout the notebook.

In [34]:
import os, tempfile
from typing import List


def write_cnf(n_vars: int, clauses: List[List[int]], comment: str = '') -> str:
    """Write a CNF formula to a temporary DIMACS file and return the path."""
    fd, path = tempfile.mkstemp(suffix='.cnf')
    with os.fdopen(fd, 'w') as f:
        if comment:
            f.write(f'c {comment}\n')
        f.write(f'p cnf {n_vars} {len(clauses)}\n')
        for clause in clauses:
            f.write(' '.join(map(str, clause)) + ' 0\n')
    return path

# ---------------------------------------
# Formulas used throughout this notebook
# ---------------------------------------
# x1 XOR x2 with a free variable x3
# 4 Models
xor_path = write_cnf(
    n_vars=3,
    clauses=[[1, 2], [-1, -2]],
    comment='x1 XOR x2 with free x3',
)

# Trivially UNSAT: x1 AND -x1
unsat_path = write_cnf(
    n_vars=1, 
    clauses=[[1], [-1]], 
    comment='UNSAT'
)

# 3-colouring, 3 variables, each pair must differ
# r=1,g=2,b=3 for node A
# r=4,g=5,b=6 for node B
# r=7,g=8,b=9 for node C
# each node gets exactly one colour & adjacent nodes differ.
colour_path = write_cnf(
    n_vars=9,
    clauses=[
        # each node gets at least one colour
        [1, 2, 3], [4, 5, 6], [7, 8, 9],
        # each node gets at most one colour
        [-1,-2],[-1,-3],[-2,-3],
        [-4,-5],[-4,-6],[-5,-6],
        [-7,-8],[-7,-9],[-8,-9],
        # adjacent nodes (A-B, B-C, A-C) differ
        [-1,-4],[-2,-5],[-3,-6],
        [-4,-7],[-5,-8],[-6,-9],
        [-1,-7],[-2,-8],[-3,-9],
    ],
    comment='3-colouring',
)

print('DIMACS file for XOR:')
with open(xor_path) as f:
    print(f.read())

DIMACS file for XOR:
c x1 XOR x2 with free x3
p cnf 3 2
1 2 0
-1 -2 0



---
## 1. The `Circuit` Object

A `Circuit` is the same klat circuit that stores all compiled DNNF nodes.
Multiple formulas can be compiled *into the same circuit*, resulting in
shared structure and efficiënt circuit inference.

| Method | Description |
| ------ | ----------- |
| `nb_nodes()` | Total number of nodes in the circuit |
| `nb_root_nodes()` | Number of roots registered via `set_root()` |
| `nb_vars()` | Number of distinct variable names |
| `set_root(nptr)` | Register a `NodePtr` as a root (prevents GC) |
| `remove_unused_nodes()` | Prune nodes not reachable from any root |
| `or_node([nptr, ...])` | Build an OR node over existing `NodePtr`s |
| `true_node()` / `false_node()` | Constant True/False sentinels |
| `literal_node(lit)` | Leaf for DIMACS literal `lit` |
| `and_node([nptr, ...])` | Build an AND node over `NodePtr`s |

In [63]:
circuit = p.Circuit()

print(f'Empty circuit:')
print(f'nb_nodes = {circuit.nb_nodes()}')
print(f'nb_root_nodes = {circuit.nb_root_nodes()}')
print(f'nb_vars = {circuit.nb_vars()}')

Empty circuit:
nb_nodes = 0
nb_root_nodes = 0
nb_vars = 0


### 1.1 Manual node construction

You can build circuit nodes directly, like in klay, useful for unit tests and
for understanding what the compilers produce under the hood.

In [65]:
circ = p.Circuit()

t   = circ.true_node()
f   = circ.false_node()
x1  = circ.literal_node(1)
nx1 = circ.literal_node(-1)
x2  = circ.literal_node(2)

# x1 AND x2
and_node = circ.and_node([x1, x2])

# x1 OR -x1
taut = circ.or_node([x1, nx1])

circ.set_root(and_node)
circ.set_root(taut)
print(f'Circuit with {circ.nb_nodes()} nodes and {circ.nb_root_nodes()} roots')
circ.print()

Circuit with 9 nodes and 2 roots
--- next layer ---
T0/1 connects to 
F0/0 connects to 
L0/2 connects to 
L0/3 connects to 
L0/4 connects to 

--- next layer ---
A1/0 connects to L0/2,L0/4,
A1/1 connects to L0/2,
A1/2 connects to L0/3,

--- next layer ---
O2/0 connects to A1/1,A1/2,



---
## 2. Compiling CNF with Different Backends

Each `compile_from_cnf_*` function takes a `Circuit` and a path to a
DIMACS `.cnf` file and returns a `NodePtr` to the compiled (sub)circuit root.

```
nptr = p.compile_from_cnf_using_ganak(circuit, path, *, ganak_options, arjun_options)
nptr = p.compile_from_cnf_using_d4v2(circuit,  path, *, options)
nptr = p.compile_from_cnf_using_sdd(circuit,   path, *, vtree_type='balanced')
nptr = p.compile_from_sdd(circuit, sdd_node)   # from a pysdd node!
```

> **Always** call `circuit.set_root(nptr)` after each `compile_from_*` call.

### 2.1 Ganak

In [37]:
print(xor_path)

/tmp/tmpc8b9q485.cnf


In [38]:
circuit = p.Circuit()

a_opts = p.ArjunOptions()
a_opts.do_arjun = False

root_xor = p.compile_from_cnf_using_ganak(circuit, xor_path, arjun_options=a_opts)
circuit.set_root(root_xor)
circuit.remove_unused_nodes()

print(f'[Ganak] XOR: {circuit.nb_nodes()} nodes, {circuit.nb_root_nodes()} roots')

[Ganak] XOR: 15 nodes, 1 roots


In [39]:
circuit.print()

--- next layer ---
T0/1 connects to 
L0/2 connects to 
L0/3 connects to 
L0/4 connects to 
L0/5 connects to 
L0/6 connects to 
L0/7 connects to 
F0/0 connects to 

--- next layer ---
A1/0 connects to L0/7,
A1/1 connects to L0/6,
A1/2 connects to L0/2,L0/5,
A1/3 connects to L0/3,L0/4,

--- next layer ---
O2/0 connects to A1/1,A1/0,
O2/1 connects to A1/3,A1/2,

--- next layer ---
A3/0 connects to O2/1,O2/0,



### 2.2 Ganak + Arjun (independent-support minimisation pre-pass)

In [40]:
circuit = p.Circuit()

a_opts = p.ArjunOptions()
a_opts.do_arjun = True

root = p.compile_from_cnf_using_ganak(circuit, xor_path, arjun_options=a_opts)
circuit.set_root(root)
circuit.remove_unused_nodes()

print(f'[Ganak+Arjun] XOR: {circuit.nb_nodes()} nodes')

[Ganak+Arjun] XOR: 15 nodes


### 2.3 SDD (Sentential Decision Diagrams)

> **Requires:** `pip install pysdd` (pysdd must be present).  
> or, `pip install 'kompyle[dev]'` can be used which includes pysdd.

You can compile from a CNF file directly, or pass a `pysdd.SddNode` you
built yourself.

In [41]:
!pip install pysdd


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [42]:
try:
    # 1) from a CNF file
    circuit = p.Circuit()
    root = p.compile_from_cnf_using_sdd(circuit, xor_path)
    circuit.set_root(root)
    circuit.remove_unused_nodes()
    print(f'[SDD from file]    XOR: {circuit.nb_nodes()} nodes')

    # 2) from a live pysdd SddNode
    from pysdd.sdd import SddManager
    mgr, sdd_node = SddManager.from_cnf_file(xor_path.encode(), vtree_type=b'balanced')

    circuit = p.Circuit()
    root = p.compile_from_sdd(circuit, sdd_node)
    circuit.set_root(root)
    circuit.remove_unused_nodes()
    print(f'[SDD from SddNode] XOR: {circuit.nb_nodes()} nodes')

except ImportError as e:
    print(f'pysdd not installed, skipping SDD cells: {e}')

Read CNF: vars=3 clauses=2
[SDD from file]    XOR: 9 nodes
Read CNF: vars=3 clauses=2
[SDD from SddNode] XOR: 9 nodes


### 2.4 d4v2

In [43]:
circuit = p.Circuit()
root = p.compile_from_cnf_using_d4v2(circuit, xor_path)
circuit.set_root(root)
circuit.remove_unused_nodes()
print(f'[d4v2] XOR: {circuit.nb_nodes()} nodes')

[d4v2] XOR: 14 nodes


c [PREPOC BACKBONE] Is running for at most 60 seconds


### 2.5 Side-by-side comparison across backends

All backends compile the same logical formula, but may produce
circuits of different sizes, neither is wrong, they just represent
the d-DNNF differently.

In [44]:
aopt = p.ArjunOptions()
aopt.do_arjun = False

backends = {
    'ganak' : lambda c, f: p.compile_from_cnf_using_ganak(c, f, arjun_options=aopt),
    'ganak+arjun': lambda c, f: p.compile_from_cnf_using_ganak(c, f),
    'd4v2' : lambda c, f: p.compile_from_cnf_using_d4v2(c, f),
}

formulas = {
    'XOR' : xor_path,
    'UNSAT' : unsat_path,
    '3-colouring': colour_path,
}

print(f"{'Formula':<18} {'Backend':<16} {'Nodes':>6} {'Roots':>6}")
print('-' * 50)

for fname, fpath in formulas.items():
    for bname, bfn in backends.items():
        circ = p.Circuit()
        nptr = bfn(circ, fpath)
        circ.set_root(nptr)
        circ.remove_unused_nodes()
        print(f'{fname:<18} {bname:<16} {circ.nb_nodes():>6} {circ.nb_root_nodes():>6}')

Formula            Backend           Nodes  Roots
--------------------------------------------------
XOR                ganak                15      1
XOR                ganak+arjun          15      1
XOR                d4v2                 14      1
UNSAT              ganak                 4      1
UNSAT              ganak+arjun           4      1
UNSAT              d4v2                  1      1
3-colouring        ganak               243      1
3-colouring        ganak+arjun         243      1
3-colouring        d4v2                 68      1


c [PREPOC BACKBONE] Is running for at most 60 seconds
c [PREPOC BACKBONE] Is running for at most 60 seconds
c [PREPOC BACKBONE] Is running for at most 60 seconds


---
## 3. `GanakOptions` and `ArjunOptions`

Pass option objects as keyword arguments to `compile_from_cnf_using_ganak`
or `count_from_cnf_using_ganak`.

### Some `GanakOptions` fields

| Field | Default | Description |
| ----- | ------- | ----------- |
| `verb` | 0 | Verbosity (0 = silent) |
| `do_restart` | False | Cube-and-conquer restarts |
| `first_restart` | None | First restart interval in conflicts |
| `maximum_cache_size_mb` | 2500 | Component cache limit |
| `polar_type` | Standard | Branch polarity |
| `do_td` | True | Tree-decomposition pre-pass |

### Some `ArjunOptions` fields

| Field | Default | Description |
| ----- | ------- | ----------- |
| `do_arjun` | False | Enable the minimisation pass |
| `verb` | 0 | Verbosity |
| `num_threads` | 1 | Parallelism inside Arjun |

In [66]:
g_opts = p.GanakOptions()
g_opts.verb = 0
g_opts.maximum_cache_size_mb = 1024
g_opts.do_td = True
g_opts.do_restart = False

a_opts = p.ArjunOptions()
g_opts.verb = 1
a_opts.do_arjun = True
a_opts.arjun_gates = True
a_opts.do_pre_backbone = True
a_opts.num_threads = 1

circuit = p.Circuit()
root = p.compile_from_cnf_using_ganak(
    circuit, colour_path,
    ganak_options=g_opts,
    arjun_options=a_opts,
)
circuit.set_root(root)
circuit.remove_unused_nodes()
print(f'3-colouring via Ganak+Arjun : {circuit.nb_nodes()} nodes')

3-colouring via Ganak+Arjun : 243 nodes
c o [backbone-simpl] ccnr sols: 10 drop_cands: 8 T: 0.00
c o [backbone-simpl] cadiback called with -- lits: 45 num cls: 21 num vars: 9
c o CadiBack BackBone Extractor
c o Copyright (c) 2023 Armin Biere University of Freiburg
c o Version 0.2.1 a44d5a94c8b8c2c4c8c77116ce80d2bb3a974252
c o CaDiCaL 2.1.3 729939aba815b1837b1590279e66c61ed9d3092f
c o Compiled with 'c++ -fPIC -fno-omit-frame-pointer -W -O3 -ggdb3 -std=c++17'
c o not checking models and backbones (enable with '--check')
c o found 9 variables
c o
c o starting solving after 2.26 seconds
c o SAT solver call 1 (9 candidates remain 100%)
c o solver determined first model after 2.26 seconds
c o SAT solver call 2 (1 candidates remain 11%)
c o
c o --- [ backbone statistics ] ------------------------------------------------
c o
c o found                 0 backbones       0%
c o dropped               9 candidates    100%
c o
c o filtered              0 candidates      0%
c o flippable             

---
## 4. `D4Options`

Pass a `D4Options` instance to `compile_from_cnf_using_d4v2`,
`compile_from_gates_formula_using_d4v2`, or `compile_from_gates_file_using_d4v2`.

### `D4PreprocMethod` enum

| Value | Description |
| ----- | ----------- |
| `Equiv` | Equivalence-based preprocessing (default) |
| `Backbone` | Backbone computation |
| `Vivi` | Vivification |
| `OccElim` | Occurrence-list elimination |
| `Comb` | Combined |
| `Basic` | Minimal |

### `D4Solver` enum

| Value | Description |
| ----- | ----------- |
| `glucose` | Glucose SAT solver |
| `minisat` | MiniSAT SAT solver |

In [46]:
d4_opts = p.D4Options()
d4_opts.preproc_method = p.D4PreprocMethod.Equiv
d4_opts.preproc_nb_iter = 1
d4_opts.preproc_timeout = 100
d4_opts.solver = p.D4Solver.glucose

circuit = p.Circuit()
root = p.compile_from_cnf_using_d4v2(circuit, colour_path, options=d4_opts)
circuit.set_root(root)
circuit.remove_unused_nodes()
print(f'3-colouring via d4v2: {circuit.nb_nodes()} nodes')

print()
print(f"{'Preproc':<12} {'Nodes':>6}")
print('-' * 20)
for method in [p.D4PreprocMethod.Equiv, 
               p.D4PreprocMethod.Basic,
               p.D4PreprocMethod.Backbone]:
    opts = p.D4Options()
    opts.preproc_method = method
    circ = p.Circuit()
    nptr = p.compile_from_cnf_using_d4v2(circ, colour_path, options=opts)
    circ.set_root(nptr)
    circ.remove_unused_nodes()
    print(f'{str(method):<12} {circ.nb_nodes():>6}')

3-colouring via d4v2: 66 nodes

Preproc       Nodes
--------------------
D4PreprocMethod.Equiv     68
D4PreprocMethod.Basic     68
D4PreprocMethod.Backbone     66


c [PREPOC BACKBONE] Is running for at most 100 seconds
c [PREPOC BACKBONE] Is running for at most 60 seconds
c [PREPOC BACKBONE] Is running for at most 60 seconds


---
## 5. The `GatedFormula` API

A `GatedFormula` lets you build a Boolean circuit *in-memory* using named
AND/OR gates and then compile it with d4v2. It maps directly to the BC-S1.2
file format (see Section 6) but lives entirely in Python.

```
gf = p.GatedFormula()
gf.add_input('in1')                         # declare input variable
gf.add_input('in2')                         # declare input variable
gf.add_and('gate1', ['in1', '-in2'])        # AND gate; '-' = negation
gf.add_or('gate2', ['in1',  'gate1'])       # OR gate
gf.add_target('gate2')                      # mark output; may also be negated
gf.display()                                # pretty-print to stdout

root = p.compile_from_gates_formula_using_d4v2(circuit, gf)
```

> A `GatedFormula` is a **pure value**, it holds no reference to any circuit
> and can be compiled into multiple circuits.

### 5.1 Basic gates

In [47]:
# x1 AND x2
gf_and = p.GatedFormula()
gf_and.add_input('x1')
gf_and.add_input('x2')
gf_and.add_and('g_and', ['x1', 'x2'])
gf_and.add_target('g_and')

circ = p.Circuit()
root = p.compile_from_gates_formula_using_d4v2(circ, gf_and)
circ.set_root(root); circ.remove_unused_nodes()
print(f'x1 AND x2: {circ.nb_nodes()} nodes')

# x1 OR -x2
gf_or = p.GatedFormula()
gf_or.add_input('x1')
gf_or.add_input('x2')
gf_or.add_or('g_or', ['x1', '-x2'])
gf_or.add_target('g_or')

circ2 = p.Circuit()
root2 = p.compile_from_gates_formula_using_d4v2(circ2, gf_or)
circ2.set_root(root2); circ2.remove_unused_nodes()
print(f'x1 OR -x2: {circ2.nb_nodes()} nodes')

x1 AND x2: 4 nodes
x1 OR -x2: 15 nodes


### 5.2 Multi-level gate network

In [48]:
# g3 = (a AND b) OR (-c AND b)
gf = p.GatedFormula()
gf.add_input('a')
gf.add_input('b')
gf.add_input('c')
gf.add_and('g1', ['a',  'b'])    # g1 := a AND b
gf.add_and('g2', ['-c', 'b'])    # g2 := -c AND b
gf.add_or('g3', ['g1', 'g2'])    # g3 := g1 OR g2
gf.add_target('g3')

circ = p.Circuit()
root = p.compile_from_gates_formula_using_d4v2(circ, gf)
circ.set_root(root); circ.remove_unused_nodes()
print(f'\nCompiled circuit: {circ.nb_nodes()} nodes')


Compiled circuit: 21 nodes


### 5.3 Helper: encode any CNF as a `GatedFormula`

In [49]:
def cnf_to_gated_formula(n_vars: int, clauses: List[List[int]]) -> p.GatedFormula:
    """Encode a CNF as an equivalent GatedFormula (AND-of-OR structure)."""
    gf = p.GatedFormula()
    for v in range(1, n_vars + 1):
        gf.add_input(str(v))

    next_id = n_vars + 1

    if not clauses:
        taut_ids = []
        for v in range(1, n_vars + 1):
            gf.add_or(str(next_id), [str(v), str(-v)])
            taut_ids.append(str(next_id))
            next_id += 1
        gf.add_or(str(next_id), taut_ids)
        gf.add_target(str(next_id))
        return gf

    clause_gate_ids = []
    for clause in clauses:
        gf.add_or(str(next_id), [str(lit) for lit in clause])
        clause_gate_ids.append(str(next_id))
        next_id += 1

    if len(clause_gate_ids) == 1:
        gf.add_target(clause_gate_ids[0])
    else:
        gf.add_and(str(next_id), clause_gate_ids)
        gf.add_target(str(next_id))
    return gf


# Encode XOR via GatedFormula
gf_xor = cnf_to_gated_formula(n_vars=3, clauses=[[1, 2], [-1, -2]])
circ_gf = p.Circuit()
root_gf = p.compile_from_gates_formula_using_d4v2(circ_gf, gf_xor)
circ_gf.set_root(root_gf); circ_gf.remove_unused_nodes()
print(f'XOR via GatedFormula: {circ_gf.nb_nodes()} nodes')

# Same formula via direct CNF path
circ_cnf = p.Circuit()
root_cnf = p.compile_from_cnf_using_d4v2(circ_cnf, xor_path)
circ_cnf.set_root(root_cnf); circ_cnf.remove_unused_nodes()
print(f'XOR via CNF path: {circ_cnf.nb_nodes()} nodes ')

XOR via GatedFormula: 14 nodes
XOR via CNF path: 14 nodes 


c [PREPOC BACKBONE] Is running for at most 60 seconds


### 5.4 Negated target

In [50]:
# add_target supports a '-' prefix to negate the output
gf_neg = p.GatedFormula()
gf_neg.add_input('x1')
gf_neg.add_input('x2')
gf_neg.add_and('g', ['x1', 'x2'])
gf_neg.add_target('-g')   # compiles -(x1 AND x2)!

circ_neg = p.Circuit()
root_neg = p.compile_from_gates_formula_using_d4v2(circ_neg, gf_neg)
circ_neg.set_root(root_neg); circ_neg.remove_unused_nodes()
print(f'x1 NAND x2: {circ_neg.nb_nodes()} nodes')

x1 NAND x2: 15 nodes


---
## 6. The `.bc` Boolean-Circuit File Format

The BC-S1.2 format describes a gate network line-by-line:

```
c  Comment line
I <var_name>                   # input (identity gate)
G <out> := A <in1> <in2> ...   # AND gate
G <out> := O <in1> <in2> ...   # OR  gate
T <output_literal>             # Target literal (may be negated: -name)
```

Compile with `compile_from_gates_file_using_d4v2(circuit, path, options=...)`.

### 6.1 Read and inspect a `.bc` file

In [51]:
bc_path = '../assets/circuits/circ1.bc'

with open(bc_path) as f:
    print(f.read())

c BC-S1.2
T root
I i1
I i2
I i0
G g0 := A i1 i2 i0
G g1 := O i1 -g0 -i2
G g2 := A -g0 i0 i1 i2
G g3 := O -g1 i2
G g4 := O -g1 -g2 -i2
G root := O g4 g3



### 6.2 Compile from a `.bc` file

In [52]:
circuit_bc = p.Circuit()
root_bc = p.compile_from_gates_file_using_d4v2(circuit_bc, bc_path)
circuit_bc.set_root(root_bc)
circuit_bc.remove_unused_nodes()
print(f'circ1.bc: {circuit_bc.nb_nodes()} nodes, {circuit_bc.nb_root_nodes()} roots')

circ1.bc: 31 nodes, 1 roots


---
## 7. Combining Circuits

A single `Circuit` can host **multiple compiled formulas** and combine their
roots with high-level operators. Sub-expressions are automatically shared.

> **Rule:** always call `set_root(nptr)` **after each** `compile_from_*` call,
> *before* the next one, or GC may collect the first sub-circuit.

### 7.1 Sequential compilation into the same arena

In [54]:
toy0_path = '../assets/toy/toy0.cnf'
toy1_path = '../assets/toy/toy1.cnf'

circuit = p.Circuit()

nptr1 = p.compile_from_cnf_using_ganak(circuit, toy0_path)
circuit.set_root(nptr1)
nb_first = circuit.nb_nodes()
print(f'After toy0: {nb_first} nodes, {circuit.nb_root_nodes()} roots')

nptr2 = p.compile_from_cnf_using_ganak(circuit, toy1_path)
circuit.set_root(nptr2)
nb_second = circuit.nb_nodes()
print(f'After toy1: {nb_second} nodes, {circuit.nb_root_nodes()} roots')
print(f'Marginal nodes from toy1: {nb_second - nb_first}')

After toy0: 226 nodes, 1 roots
After toy1: 286 nodes, 2 roots
Marginal nodes from toy1: 60


### 7.2 OR of two compiled formulas

In [55]:
# Build F1 OR F2 as a single OR node
nptr_or = circuit.or_node([nptr1, nptr2])
circuit.set_root(nptr_or)
print(f'After or_node: {circuit.nb_nodes()} nodes, {circuit.nb_root_nodes()} roots')

After or_node: 304 nodes, 3 roots


### 7.3 AND of two compiled formulas

In [56]:
# Build F1 AND F2 as a single AND node
circuit2 = p.Circuit()
n1 = p.compile_from_cnf_using_ganak(circuit2, toy0_path)
circuit2.set_root(n1)
n2 = p.compile_from_cnf_using_ganak(circuit2, toy1_path)
circuit2.set_root(n2)

nptr_and = circuit2.and_node([n1, n2])
circuit2.set_root(nptr_and)
print(f'F1 AND F2: {circuit2.nb_nodes()} nodes')

F1 AND F2: 302 nodes


### 7.4 Garbage collection with `remove_unused_nodes`

In [57]:
circ_gc = p.Circuit()
n_a = p.compile_from_cnf_using_ganak(circ_gc, toy0_path)
# intentionally NOT calling set_root(n_a) yet
n_b = p.compile_from_cnf_using_ganak(circ_gc, toy1_path)
# only register n_b
circ_gc.set_root(n_b)

before = circ_gc.nb_nodes()
circ_gc.remove_unused_nodes()
after = circ_gc.nb_nodes()
print(f'Before pruning: {before} nodes')
print(f'After pruning:  {after} nodes  (toy0 sub-circuit removed)')

Before pruning: 286 nodes
After pruning:  78 nodes  (toy0 sub-circuit removed)


---
## 8. Model Counting Without Compilation

When you only need the model count (not the circuit), use the dedicated
count functions, they skip circuit construction entirely and return an
arbitrary-precision Python `int`.

```python
count = p.count_from_cnf_using_ganak(cnf_file, ganak_options=..., arjun_options=..., weighted_counting=False)
count = p.count_from_cnf_using_d4v2(cnf_file, options=...)
count = p.count_from_cnf_using_sdd(cnf_file, vtree_type='balanced')
```

In [58]:
# x1 XOR x2 with free x3 has 4 models
c_ganak = p.count_from_cnf_using_ganak(xor_path)
c_d4    = p.count_from_cnf_using_d4v2(xor_path)
print(f'XOR model count (Ganak): {c_ganak}')
print(f'XOR model count (d4v2) : {c_d4}')
assert c_ganak == c_d4 == 4

# UNSAT has 0 models
assert p.count_from_cnf_using_ganak(unsat_path) == 0
print(f'UNSAT count: {p.count_from_cnf_using_ganak(unsat_path)}')

# 3-colouring of a triangle has 6 models
c_col = p.count_from_cnf_using_ganak(colour_path)
print(f'3-colouring count (Ganak): {c_col}')

XOR model count (Ganak): 4
XOR model count (d4v2) : 4
UNSAT count: 0
3-colouring count (Ganak): 6


c [PREPOC BACKBONE] Is running for at most 60 seconds


### 8.1 Weighted model counting with Ganak

In [59]:
# weighted_counting=True activates Ganak's WMC mode.
# This feature was used in order to find out if ganak has any compute overhead compared to MC.
c_weighted = p.count_from_cnf_using_ganak(xor_path, weighted_counting=True)
print(f'XOR weighted count (Ganak): {c_weighted}')

XOR weighted count (Ganak): 4


---
## 9. Validating the Result: `check_sdnnf`

After compiling, you can verify the output satisfies the d-DNNF
properties decomposability *and* smoothness:

```python
result = p.check_sdnnf(circuit)
result.violations
```

You can also check properties individually:
```python
p.check_decomposability(circuit, max_violations=10)
p.check_smooth(circuit, max_violations=10)
```

In [60]:
circuit = p.Circuit()
root = p.compile_from_cnf_using_ganak(circuit, xor_path)
circuit.set_root(root)
circuit.remove_unused_nodes()

result = p.check_sdnnf(circuit)
print(f'check_sdnnf is_decomposable: {result.is_decomposable}')
print(f'check_sdnnf is_smooth: {result.is_smooth}')
if result.violations:
    for v in result.violations:
        print(f'  violation: {v}')

dec = p.check_decomposability(circuit, max_violations=10)
smo = p.check_smooth(circuit, max_violations=10)
print(f'check_decomposability: {dec.is_decomposable}')
print(f'check_smooth: {smo.is_smooth}')

check_sdnnf is_decomposable: True
check_sdnnf is_smooth: True
check_decomposability: True
check_smooth: True


In [61]:
# Verify all backends produce valid s-d-DNNF
print(f"{'Backend':<16} {'OK':>4} {'Nodes':>6}")
print('-' * 30)
for name, fn in [
    ('ganak', lambda c, f: p.compile_from_cnf_using_ganak(c, f)),
    ('d4v2', lambda c, f: p.compile_from_cnf_using_d4v2(c, f)),
]:
    circ = p.Circuit()
    nptr = fn(circ, colour_path)
    circ.set_root(nptr)
    circ.remove_unused_nodes()
    ok = result.is_decomposable and result.is_smooth
    print(f'{name:<16} {str(ok):>4} {circ.nb_nodes():>6}')

Backend            OK  Nodes
------------------------------
ganak            True    243
d4v2             True     68


c [PREPOC BACKBONE] Is running for at most 60 seconds


---
## 10. Performance Stats

Kompyle exposes per-operation counters and nanosecond timers for both Ganak
and d4v2. These are useful for measuring the overhead of building the klay
circuit versus pure counting.

| Function | Description |
| -------- | ----------- |
| `get_ganak_stats()` | Returns `{'circuit': {...}, 'count': {...}}` |
| `get_d4_stats()` | Same for d4v2 |
| `reset_ganak_stats()` | Zero both Ganak stat globals |
| `reset_d4_stats()` | Zero both d4 stat globals |
| `reset_all_stats()` | Zero all four stat globals |

Each inner dict maps `n_*` (call count) and `ns_*` (total nanoseconds) keys.
Subtract `count` from `circuit` per key to get the klay-specific overhead.

In [62]:
p.reset_all_stats()

circ = p.Circuit()
r = p.compile_from_cnf_using_d4v2(circ, colour_path)
circ.set_root(r)
circ.remove_unused_nodes()
stats_compile = p.get_d4_stats()

p.reset_d4_stats()

p.count_from_cnf_using_d4v2(colour_path)
stats_count = p.get_d4_stats()

print('--- d4v2 stats (circuit path) ---')
for k, v in stats_compile['circuit'].items():
    print(f'  circuit.{k:20s} = {v}')

print()
print('--- d4v2 stats (count-only path) ---')
for k, v in stats_count['count'].items():
    print(f'  count.{k:22s} = {v}')

print()
print('--- Klay overhead (ns_add, ns_mul) ---')
for key in ['ns_add', 'ns_mul']:
    circ_val  = stats_compile['circuit'].get(key, 0)
    count_val = stats_count['count'].get(key, 0)
    print(f'  overhead {key}: {circ_val - count_val} ns')

--- d4v2 stats (circuit path) ---
  circuit.n_top                = 6
  circuit.ns_top               = 2654
  circuit.n_bottom             = 0
  circuit.ns_bottom            = 0
  circuit.n_branch             = 1
  circuit.ns_branch            = 407
  circuit.n_add                = 5
  circuit.ns_add               = 42008
  circuit.n_mul                = 5
  circuit.ns_mul               = 283
  circuit.n_lit_node           = 41
  circuit.ns_lit_node          = 6730
  circuit.n_taut               = 0
  circuit.ns_taut              = 0

--- d4v2 stats (count-only path) ---
  count.n_top                  = 6
  count.ns_top                 = 241
  count.n_bottom               = 0
  count.ns_bottom              = 0
  count.n_branch               = 1
  count.ns_branch              = 146
  count.n_add                  = 5
  count.ns_add                 = 7655
  count.n_mul                  = 5
  count.ns_mul                 = 380
  count.n_lit_node             = 0
  count.ns_lit_node          

c [PREPOC BACKBONE] Is running for at most 60 seconds
c [PREPOC BACKBONE] Is running for at most 60 seconds


---
## 11. API Quick Reference

### Compile to circuit
```python
import kompyle as p

circuit = p.Circuit()

# CNF file  to  klay circuit
nptr = p.compile_from_cnf_using_ganak(circuit, 'file.cnf',
           ganak_options=p.GanakOptions(), arjun_options=p.ArjunOptions())
nptr = p.compile_from_cnf_using_d4v2(circuit, 'file.cnf', options=p.D4Options())
nptr = p.compile_from_cnf_using_sdd(circuit, 'file.cnf', vtree_type='balanced')

# pysdd node  to  klay circuit
nptr = p.compile_from_sdd(circuit, sdd_node)

# GatedFormula  to  klay circuit
gf = p.GatedFormula()
gf.add_input('x1'); gf.add_input('x2')
gf.add_and('g', ['x1', 'x2']); gf.add_target('g')
nptr = p.compile_from_gates_formula_using_d4v2(circuit, gf, options=p.D4Options())

# .bc file  to  klay circuit
nptr = p.compile_from_gates_file_using_d4v2(circuit, 'file.bc', options=p.D4Options())
```

### Count only (no circuit)
```python
count = p.count_from_cnf_using_ganak(cnf_file, ganak_options=..., arjun_options=...,
                                      weighted_counting=False)
count = p.count_from_cnf_using_d4v2(cnf_file, options=...)
count = p.count_from_cnf_using_sdd(cnf_file, vtree_type='balanced')
```

### Circuit management
```python
circuit.set_root(nptr)
circuit.remove_unused_nodes()
circuit.nb_nodes()
circuit.nb_root_nodes()
circuit.nb_vars()
circuit.or_node([nptr1, nptr2])
circuit.and_node([nptr1, nptr2])
circuit.literal_node(lit)
circuit.true_node()
circuit.false_node()
```

### Validation
```python
p.check_sdnnf(nptr)
p.check_decomposability(nptr)
p.check_smooth(nptr)
result.violations
```

### Options
```python
g = p.GanakOptions()
g.polar_type = p.GanakPolarType.Cache
g.do_restart = True
g.maximum_cache_size_mb = 2500

a = p.ArjunOptions()
a.do_arjun = True
a.num_threads = 4

d = p.D4Options()
d.preproc_method = p.D4PreprocMethod.Equiv
d.solver = p.D4Solver.glucose
```

### Stats
```python
p.reset_all_stats()
p.get_ganak_stats()
p.get_d4_stats()
```